In [1]:
!pip install -q transformers accelerate
!pip install -q peft trl bitsandbytes
!pip install -q datasets evaluate bert-score
!pip install -q gradio mlflow wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 73.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

In [3]:
bnb_config  = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [4]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

base_model_name = "meta-llama/Llama-3.2-3B"
adapter_path = "/content/drive/MyDrive/llm-lora-finetuning/lora-adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
base_model.eval()

# Fine-tuned model
finetuned_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)
finetuned_model = PeftModel.from_pretrained(finetuned_model, adapter_path)
finetuned_model.eval()

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 3072)
        (layers): ModuleList(
          (0-27): 28 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linea

In [5]:
from datasets import load_from_disk, load_dataset

In [6]:
ds = load_dataset("sahil2801/CodeAlpaca-20k")
split = ds['train'].train_test_split(test_size=0.1, seed=42)
ds_test = split['test']

README.md:   0%|          | 0.00/147 [00:00<?, ?B/s]

code_alpaca_20k.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/20022 [00:00<?, ? examples/s]

In [7]:
def sample_to_string(sample: dict) -> str:
  instruction, input, output = sample["instruction"], sample["input"], sample["output"]
  if input == "":
    return f"""### Instruction:
                {instruction}

                ### Response:
                {output}"""
  else:
    return f"""### Instruction:
                {instruction}

                ### Input:
                {input}

                ### Response:
                {output}"""

# New function to format samples for the dataset, creating a 'text' column
def formatting_func(sample: dict) -> dict:
  return {"text": sample_to_string(sample)}

# Apply the formatting function to the entire dataset, creating a 'text' column
ds_test = ds_test.map(formatting_func, batched=False)

# Display a sample of the formatted text from the training split for verification
print(ds_test[0]["text"])

Map:   0%|          | 0/2003 [00:00<?, ? examples/s]

### Instruction:
                Design a class for representing a person in Python.

                ### Response:
                class Person:
    """
    Class to represent a person
    """
    def __init__(self, name, age, address):
        self.name = name
        self.age = age
        self.address = address
    
    def birthday(self):
        """
        Increments the age of the person
        """
        self.age += 1


In [8]:
def run_inference(model, tokenizer, row):
  test_prompt = sample_to_string(row)
  test_prompt = test_prompt.split("### Response")[0]
  test_prompt += "\n### Response:\n"                  # ← FIXED: was \\n (escaped), now real newline

  input_tokenized = tokenizer(
      test_prompt,
      return_tensors="pt",
      truncation=True,
      max_length=512
  ).to("cuda")

  result = model.generate(
      **input_tokenized,
      max_new_tokens=200,
      do_sample=False,
      pad_token_id=tokenizer.eos_token_id
  )
  generated_only = result[0][input_tokenized["input_ids"].shape[1]:]
  return tokenizer.decode(generated_only, skip_special_tokens=True)

In [9]:
from tqdm import tqdm

base_outputs, finetuned_outputs, references = [], [], []

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token  # required for Llama models

for row in tqdm(ds_test.select(range(200))):
    base_outputs.append(run_inference(base_model, tokenizer, row))
    finetuned_outputs.append(run_inference(finetuned_model, tokenizer, row))
    references.append(row["output"])   # ground truth answer

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

  2%|▏         | 4/200 [01:14<1:00:47, 18.61s/it]


KeyboardInterrupt: 

In [ ]:
import json
with open("/content/drive/MyDrive/llm-lora-finetuning/eval_outputs.json", "w") as f:
    json.dump({
        "base_outputs": base_outputs,
        "finetuned_outputs": finetuned_outputs,
        "references": references
    }, f)
print("Outputs saved!")

In [ ]:
!pip install rouge_score

In [ ]:
import evaluate
rouge = evaluate.load("rouge")

base_rouge = rouge.compute(
    predictions=base_outputs,
    references=references,
    use_aggregator=True
)
ft_rouge = rouge.compute(
    predictions=finetuned_outputs,
    references=references,
    use_aggregator=True
)

print(f"Base ROUGE-L:       {base_rouge['rougeL']:.4f}")
print(f"Fine-tuned ROUGE-L: {ft_rouge['rougeL']:.4f}")
print(f"Delta:              +{ft_rouge['rougeL'] - base_rouge['rougeL']:.4f}")

In [ ]:
from bert_score import score as bert_score

_, _, F1_base = bert_score(
    base_outputs, references,
    lang="en",
    model_type="distilbert-base-uncased",
    verbose=True
)
_, _, F1_ft = bert_score(
    finetuned_outputs, references,
    lang="en",
    model_type="distilbert-base-uncased",
    verbose=True
)

print(f"Base BERTScore F1:       {F1_base.mean().item():.4f}")
print(f"Fine-tuned BERTScore F1: {F1_ft.mean().item():.4f}")
print(f"Delta:                   +{F1_ft.mean().item() - F1_base.mean().item():.4f}")

In [ ]:
import pandas as pd
import mlflow

# Results table
df = pd.DataFrame({
    "Model":        ["Base", "Fine-tuned", "Delta"],
    "ROUGE-L":      [round(base_rouge['rougeL'], 4),
                     round(ft_rouge['rougeL'], 4),
                     round(ft_rouge['rougeL'] - base_rouge['rougeL'], 4)],
    "BERTScore-F1": [round(F1_base.mean().item(), 4),
                     round(F1_ft.mean().item(), 4),
                     round(F1_ft.mean().item() - F1_base.mean().item(), 4)]
})
print(df.to_string(index=False))

# Save to Drive
df.to_csv("/content/drive/MyDrive/llm-lora-finetuning/eval_results.csv", index=False)

# Log to MLflow
with mlflow.start_run(run_name="llama3-lora-codealpaca"):
    mlflow.log_params({
        "model": "meta-llama/Llama-3.2-3B",
        "dataset": "CodeAlpaca-20k",
        "lora_r": 8,
        "lora_alpha": 16,
        "learning_rate": 5e-5,
        "epochs": 1,
        "train_samples": 18000
    })
    mlflow.log_metrics({
        "base_rougeL":        base_rouge['rougeL'],
        "finetuned_rougeL":   ft_rouge['rougeL'],
        "base_bertscore_f1":  F1_base.mean().item(),
        "ft_bertscore_f1":    F1_ft.mean().item(),
    })
print("Logged to MLflow!")

In [14]:
finetuned_model.push_to_hub("QLoRA-Finetuning")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:  13%|#3        | 1.23MB / 9.19MB            

CommitInfo(commit_url='https://huggingface.co/parthtamu/QLoRA-Finetuning/commit/ee894328e74cbb80a80f45eea5c849c06f49125d', commit_message='Upload model', commit_description='', oid='ee894328e74cbb80a80f45eea5c849c06f49125d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/parthtamu/QLoRA-Finetuning', endpoint='https://huggingface.co', repo_type='model', repo_id='parthtamu/QLoRA-Finetuning'), pr_revision=None, pr_num=None)